In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

import sys
sys.path.extend(["../../"])

from utils import squared_loss_function, create_computation_graph_linear
from utils import activation_function_linear, activation_function_sigmoid
from utils import activation_function_relu, activation_function_softplus, activation_function_exponential
from utils import computation_graph_relu, computation_graph_softplus, computation_graph_exponential
from utils import grad_squared_loss_wrt_relu_model, grad_squared_loss_wrt_softplus_model, grad_squared_loss_wrt_exponential_model
from utils import heteroscedastic_loss_function
from utils import computation_graph_heteroscedastic_gaussian_softplus, grad_heteroscedastic_loss_wrt_softplus_model

$$
\newcommand{\lvec}{\mathbf{l}}
\newcommand{\xvec}{\mathbf{x}}
\newcommand{\xvect}{\mathbf{x}^T}
\newcommand{\Xmat}{\mathbf{X}}
\newcommand{\Xmatt}{\mathbf{X}^T}
\newcommand{\Amat}{\mathbf{A}}
\newcommand{\Amatt}{\mathbf{A}^T}
\newcommand{\avec}{\mathbf{a}}
\newcommand{\avect}{\mathbf{a}^T}
\newcommand{\Bmat}{\mathbf{B}}
\newcommand{\Bmatt}{\mathbf{B}^T}
\newcommand{\Cmat}{\mathbf{C}}
\newcommand{\cvec}{\mathbf{c}}
\newcommand{\Cmatt}{\mathbf{C}^T}
\newcommand{\tvec}{\mathbf{t}}
\newcommand{\tvect}{\mathbf{t}^T}
\newcommand{\Tmat}{\mathbf{T}}
\newcommand{\Tmatt}{\mathbf{T}^T}
\newcommand{\yvec}{\mathbf{y}}
\newcommand{\Ymat}{\mathbf{Y}}
\newcommand{\Ymatt}{\mathbf{Y}^T}
\newcommand{\Zmat}{\mathbf{Z}}
\newcommand{\zvec}{\mathbf{z}}
\newcommand{\wvec}{\mathbf{w}}
\newcommand{\wvect}{\mathbf{w}^T}
\newcommand{\wvecsigma}{\mathbf{w}_\sigma}
\newcommand{\wvecsigmat}{\mathbf{w}_{\sigma}^T}
\newcommand{\sigmatwovec}{\boldsymbol{\sigma^2}}
\newcommand{\Wmat}{\mathbf{W}}
\newcommand{\Wmatt}{\mathbf{W}^T}
\newcommand{\Wmatsigma}{\mathbf{W}_\sigma}
\newcommand{\Wmatsigmat}{\mathbf{W}_{\sigma}^T}
\newcommand{\Vmat}{\mathbf{V}}
\newcommand{\Vmatt}{\mathbf{V}^T}
\newcommand{\Vvec}{\mathbf{v}}
\newcommand{\Vvect}{\mathbf{v}^T}
\newcommand{\Imat}{\mathbf{I}}
\newcommand{\Sigmainv}{\Sigma^{-1}}
\newcommand{\onevec}{\mathbf{1}}
\newcommand{\onevect}{\mathbf{1}^T}
\newcommand{\dd}{\mathrm{d}}
\newcommand{\diag}{\text{diag}}
\newcommand{\pareinv}[1]{\left(#1\right)^{-1}}
\newcommand{\pare}[1]{\left(#1\right)}
\newcommand{\pareT}[1]{\left(#1\right)^{T}}
\renewcommand{\bra}[1]{\left[#1\right]}
\newcommand{\braT}[1]{\left[#1\right]^{T}}
\newcommand{\tr}[1]{\text{tr}\left(#1\right)}
\newcommand{\vvec}{\text{vec} }
\newcommand{\sgn}{\operatorname{sgn}}
$$

# Generalized Linear Models

This chapter covers generalized linear models. However, I will not approach this topic from the classical perspective, https://en.wikipedia.org/wiki/Generalized_linear_model, but from a more general perspective.

We have already seen that (around 95% of) machine learning can be elegantly framed through, or is very related to, parameterizing probability distributions over the data we want to model. For the moment, we have focused on the Gaussian, Laplace, and Student-t observation models. The classical viewpoint of GLMs is that we are trying to model the parameters of the probability distributions in the exponential family. 

While this is the classical way, I like to generalize this by simply parameterizing any probability distribution we want. For instance, we have seen that we can parameterize the Student-t distribution as well.

Another generalization I cover relates to restricting the model within a range of plausible values. For instance, while the GLM over the Gaussian considers predicting the mean of the distribution, we might know, from our prior knowledge, that this mean can only be within one possible range. For example, if we are targeting sensor measurements, and we know these sensor measurements provide values between $[-1,1]$, there is no sense in targeting values outside this range.

## Introduction

So far, we have seen models that are linear in the parameters, which means that we can compute the outputs given the inputs through a matrix product. Let's start with single output variables:

$$
\begin{split}
y = \xvect\wvec
\end{split}
$$

What if, for whatever reason, our predictions are only positive?. Think, for example, of a problem in which we are predicting house pricing. The equation $y = \xvect\wvec$ can make negative predictions on the price. While we could post-process the output to convert negative prices into null values, it makes much more sense to predict only positive values. This example can be extended to whatever you think. For example, we might want to predict just negative values.

This is where the concept of a link function comes into play. Link functions map the output of our model (in our case a linear (basis function) model) into the desired value for our task. We know, in reality, that this is the desired value of the parameter of the distribution modelling our task.

In general, we will be using link functions to specify the range of values we want the parameter to take for two reasons:

1) We know the outputs of our problem are restricted to an interval, as in the house price example.
2) The parameter can only take some values, as with the variance of a Gaussian observation model, or the parameter of a Bernoulli distribution in the sigmoid classification problem.



## Link functions.

Let $x$ denote the unrestricted output of our linear (basis function) model, which we will call the linear predictor. A link function $\Phi(\cdot)$ maps $x$ into the domain $\mathcal{D}$ required by the parameter we are targeting, i.e. $\theta = \Phi(x)$, $\theta \in \mathcal{D}$. Below are some of the most common ones, together with their typical domain and where you will see them used in practice.

### Identity link

$$
\Phi(x) = x, \qquad \mathcal{D} = \mathbb{R}
$$

The trivial link. Used whenever the parameter is itself unrestricted, e.g., the mean of a Gaussian observation model, as we have already seen.

### Exponential link

$$
\Phi(x) = \exp(x), \qquad \mathcal{D} = (0,\infty)
$$

Maps to strictly positive values. This is the canonical choice for the rate parameter of a Poisson distribution (classical Poisson GLM), and it is also a natural candidate for a positive scale/variance parameter, e.g., in heteroscedastic regression where we let the variance of the Gaussian likelihood depend on the input. Note, however, that in that case it is common practice to instead predict $\log \sigma^2$ directly with the identity link, and only exponentiate at the very end to recover $\sigma^2$; this is mathematically equivalent to using an exponential link, but is usually preferred for numerical stability (large activations under `exp` can overflow, while working in log-space does not). 
### Softplus

$$
\Phi(x) = \log\pare{1+\exp(x)}, \qquad \mathcal{D} = (0,\infty)
$$

A smooth, everywhere-differentiable relaxation of the ReLU (see below) that also maps to positive values. It grows linearly for large $x$ rather than exploding like `exp`, which makes it a popular choice for positive scale parameters inside neural networks (again, heteroscedastic variance/std-dev heads are a typical use case) when one wants to avoid both the numerical blow-up of the exponential link and the dead zones of the ReLU.

### ReLU

$$
\Phi(x) = \max(0,x), \qquad \mathcal{D} = [0,\infty)
$$

Also restricts the output to be non-negative, exactly as in the house-pricing example above. It is extremely common as an *internal* activation in deep networks, but it is a less common choice as the final link on top of a distribution parameter, since it is non-differentiable at $x=0$ and can get "stuck" at exactly $0$ (dead units), which is problematic if the parameter is not allowed to be exactly at the boundary of $\mathcal{D}$ (e.g. a variance of exactly $0$).

### Negative-valued links

Symmetrically, sometimes we know the parameter must be *negative*, e.g. a depth below sea level, a deficit, or a log-return that can only decrease. Given any link $\Phi_+$ that maps onto $(0,\infty)$, the mirrored map $\Phi_-(x) = -\Phi_+(-x)$ maps onto $(-\infty,0)$. Applying this to the three links above gives:

$$
\Phi(x) = \min(0,x) \quad \text{(negative ReLU)}, \qquad
\Phi(x) = -\exp(-x) \quad \text{(negative exponential)}, \qquad
\Phi(x) = -\log\pare{1+\exp(-x)} \quad \text{(negative softplus)}
$$

all with $\mathcal{D} = (-\infty,0]$ or $(-\infty,0)$ depending on whether the ReLU-style hard clip at the boundary is included. The same trade-offs discussed above for their positive counterparts apply here (numerical stability of softplus vs. exponential, non-differentiability and dead units for the ReLU case), just mirrored around $0$.

### Sigmoid (logistic function)

$$
\Phi(x) = \sigma(x) = \frac{1}{1+\exp(-x)}, \qquad \mathcal{D} = (0,1)
$$

Maps to the open unit interval, which makes it the canonical link for the parameter of a Bernoulli (or Binomial) distribution. This is exactly what you already know as logistic regression.

### Gaussian CDF (probit function)

$$
\Phi(x) = \int_{-\infty}^{x} \mathcal{N}(z; 0, 1)\, \dd z, \qquad \mathcal{D} = (0,1)
$$

Just like the sigmoid, this maps $\mathbb{R} \to (0,1)$ and can be used as an alternative link for a Bernoulli parameter; this gives rise to probit regression, historically the original GLM link for binary data before the logistic function became popular. The two curves are very similar in shape but not identical (the probit has lighter tails than the sigmoid). Here the generic link symbol $\Phi$ and the conventional notation for the Gaussian CDF happen to coincide: this is the one link where that is not a coincidence, but the very definition of the function.

The reason $\Phi$ shows up so much beyond probit regression, and in particular in Gaussian Processes classification, is analytical convenience rather than any conceptual superiority over the sigmoid: convolving a Gaussian CDF with a Gaussian density has a closed form,
$$
\int \Phi(x)\, \mathcal{N}(x; \mu, \sigma^2)\, \dd x = \Phi\pare{\frac{\mu}{\sqrt{1+\sigma^2}}},
$$
whereas the analogous integral against a sigmoid does not have a closed form. This is precisely why the probit, rather than the sigmoid, is the workhorse link in GP classification, Bayesian probit models, and other settings where we need to marginalize a Gaussian-distributed linear predictor through the link.

### Complementary log-log (cloglog)

$$
\Phi(x) = 1 - \exp\pare{-\exp(x)}, \qquad \mathcal{D} = (0,1)
$$

A third classical alternative to sigmoid and probit for the parameter of a Bernoulli distribution, available in most GLM software as the `cloglog` link. Unlike sigmoid and probit, which are symmetric around $x=0$ (i.e. $\Phi(x) = 1-\Phi(-x)$), the cloglog link is asymmetric: it approaches $0$ much more slowly than it approaches $1$. It arises as the CDF of the Gumbel (extreme-value) distribution rather than the logistic or Gaussian, which makes it the natural choice when the event being modeled is best thought of as some underlying extreme-value process crossing a threshold, e.g., discrete-time survival/hazard models, or species presence/absence data in ecology.

### tanh and general bounded intervals

$$
\Phi(x) = \tanh(x), \qquad \mathcal{D} = (-1,1)
$$

Useful whenever the target parameter is known to live in a symmetric bounded range, like the sensor example from the introduction. For an arbitrary interval $\mathcal{D} = (l,u)$ we simply rescale any of the two $(0,1)$-valued or $(-1,1)$-valued links above, e.g.
$$
\Phi(x) = l + (u-l)\,\sigma(x), \qquad \text{or} \qquad \Phi(x) = \frac{l+u}{2} + \frac{u-l}{2}\tanh(x).
$$

### Softmax

For a $K$-dimensional linear predictor $\mathbf{x} = (x_1,\dots,x_K)$,
$$
\Phi(\mathbf{x})_k = \frac{\exp(x_k)}{\sum_{j=1}^K \exp(x_j)}, \qquad \mathcal{D} = \Delta^{K-1} \; \text{(the probability simplex)}
$$

The multivariate generalization of the sigmoid: it is the canonical link for the parameter vector of a Categorical/Multinomial distribution, i.e., multi-class classification.

### Heaviside / sign function

$$
\Phi(x) = \mathbb{1}[x>0], \qquad \mathcal{D} = \{0,1\}
$$

The "hard" limit of the sigmoid as its slope goes to infinity (equivalently, $\Phi(x) = \lim_{\beta\to\infty}\sigma(\beta x)$). It is what the classical Perceptron uses to threshold its linear predictor into a class decision, but it is not differentiable, so it is not usable as a link function trained by gradient-based methods. It is worth keeping in mind, though, precisely because it clarifies why we bother with the sigmoid/probit in the first place: both are smooth, gradient-friendly relaxations of this hard threshold.

### On choosing among link functions

Several of the links above target the exact same domain (sigmoid, probit, cloglog and the shifted-and-scaled tanh all map onto $(0,1)$; ReLU, softplus and the exponential link all map onto $(0,\infty)$; their negative-valued mirrors all map onto $(-\infty,0)$). Classical exponential-family GLM theory singles out one *canonical* link per distribution: the one that makes the natural parameter of the exponential family coincide with the linear predictor, but nothing forces us to use it. In practice, the choice between links with the same domain is often a matter of convenience rather than correctness: probit over sigmoid because it makes a downstream integral tractable (as in GPs above), cloglog over both when the underlying process is better modeled as extreme-value rather than Gaussian/logistic, softplus over exponential because it is numerically better-behaved, and so on. What matters is only that $\Phi$ maps into the correct domain $\mathcal{D}$ of the parameter you are modelling. 

### Visualizing the link functions

Let's plot the link functions above over a range of the linear predictor $x$, one panel per function (or pair of mirrored functions) in a $3\times3$ grid. Each panel is a genuine scalar map $\Phi:\mathbb{R}\to\mathcal{D}$, evaluated over a grid of $x$-values.

In [ ]:
x = np.linspace(-6, 6, 400)
x_pos = np.linspace(-4, 4, 400)
LW = 2.5

fig, axes = plt.subplots(3, 3, figsize=(14, 11))

axes[0, 0].plot(x, activation_function_linear(x), color="C0", linewidth=LW)
axes[0, 0].set_title("Identity")

axes[0, 1].plot(x_pos, activation_function_exponential(x_pos), label=r"$\exp(x)$", linewidth=LW)
axes[0, 1].plot(x_pos, -activation_function_exponential(-x_pos), label=r"$-\exp(-x)$", linestyle="--", linewidth=LW)
axes[0, 1].set_ylim([-10, 10])
axes[0, 1].set_title("Exponential")
axes[0, 1].legend(fontsize=8)

axes[0, 2].plot(x_pos, activation_function_softplus(x_pos), label=r"$\log(1+e^{x})$", linewidth=LW)
axes[0, 2].plot(x_pos, -activation_function_softplus(-x_pos), label=r"$-\log(1+e^{-x})$", linestyle="--", linewidth=LW)
axes[0, 2].set_ylim([-10, 10])
axes[0, 2].set_title("Softplus")
axes[0, 2].legend(fontsize=8)

axes[1, 0].plot(x, activation_function_relu(x), label=r"$\max(0,x)$", linewidth=LW)
axes[1, 0].plot(x, -activation_function_relu(-x), label=r"$\min(0,x)$", linestyle="--", linewidth=LW)
axes[1, 0].set_title("ReLU")
axes[1, 0].legend(fontsize=8)

axes[1, 1].plot(x, activation_function_sigmoid(x), color="C3", linewidth=LW)
axes[1, 1].set_title("Sigmoid")

axes[1, 2].plot(x, norm.cdf(x), color="C4", linewidth=LW)
axes[1, 2].set_title("Probit (Gaussian CDF)")

axes[2, 0].plot(x, np.tanh(x), color="C5", linewidth=LW)
axes[2, 0].set_title("tanh")

axes[2, 1].plot(x, (x > 0).astype(float), color="black", linewidth=LW)
axes[2, 1].set_title("Heaviside")

axes[2, 2].plot(x, 1 - np.exp(-np.exp(x)), color="C6", linewidth=LW)
axes[2, 2].set_title("Complementary log-log")

for ax in axes.flat:
    ax.set_xlabel("x")
    ax.set_ylabel(r"$\Phi(x)$")
    ax.grid(True)

fig.suptitle("Link functions", fontsize=14)
plt.tight_layout()
plt.show()

## Example: restricting to positive mean

We saw in the [previous chapter](2_Linear_basis_function_models_ovefit_reg.ipynb) that a skewed, strictly positive target can sometimes be rescued by a plain linear model after taking $\log$: if the underlying noise is multiplicative, $\log$ turns it into additive, constant-variance noise, and the mean becomes linear in $\log$-space too. That is not a general recipe, though; it happened to work for `price` because the generative process behind it really was multiplicative and log-linear. Here we build a case where it is not.

Below, `sales` is generated conditional on `temperature` with perfectly Gaussian, homoscedastic noise; so, conditionally, there is nothing pathological about the noise itself. But the mean follows a softplus (not exponential) function of `temperature`. Let's check whether $\log$, a $z$-score, or the combination of both rescue this case the same way they did for `price`.

Here is the distribution of the dataset:

In [ ]:
rng = np.random.default_rng(7)
n = 300
temperature = rng.uniform(-20, 30, size=n)

w_true, b_true, sigma = 0.15, 0.5, 0.3
mean_sales = computation_graph_softplus(temperature.reshape(-1, 1), np.array([[w_true]]), np.array([[b_true]])).ravel()
sales = mean_sales + rng.normal(0, sigma, size=n)

print(f"{np.mean(sales <= 0):.0%} of the sample is non-positive (log undefined there)")


plt.scatter(temperature, sales, marker="x")
plt.xlabel("temperature")
plt.ylabel("sales")
plt.title("Data to be modeled: sales vs. temperature", fontsize=14)
plt.tight_layout()
plt.show()

Here is the distribution of the output with different normalization strategies:

In [ ]:
sales_z = (sales - sales.mean()) / sales.std()

with np.errstate(invalid="ignore"):
    log_sales = np.log(sales)
valid = ~np.isnan(log_sales)
log_sales_z = np.full_like(log_sales, np.nan)
log_sales_z[valid] = (log_sales[valid] - log_sales[valid].mean()) / log_sales[valid].std()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].hist(sales, bins=30, density=True, color="C0", edgecolor="black", alpha=0.7)
axes[0, 0].set_title("raw")

axes[0, 1].hist(sales_z, bins=30, density=True, color="C1", edgecolor="black", alpha=0.7)
axes[0, 1].set_title("z-score")

axes[1, 0].hist(log_sales[valid], bins=30, density=True, color="C2", edgecolor="black", alpha=0.7)
axes[1, 0].set_title("log")

axes[1, 1].hist(log_sales_z[valid], bins=30, density=True, color="C3", edgecolor="black", alpha=0.7)
axes[1, 1].set_title("log + z-score: still not Gaussian")

for ax in axes.flat:
    ax.grid(True)

fig.suptitle("Normalizing sales (marginal, ignoring temperature)", fontsize=14)
plt.tight_layout()
plt.show()

Here is the conditional distribution of the different normalization strategies. We see that log plus zscore turns the model into a linear model with heteroscedastic noise. This could be a modelling option.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].scatter(temperature, sales, marker="x", color="C0")
axes[0, 0].set_xlabel("temperature")
axes[0, 0].set_ylabel("sales")
axes[0, 0].set_title("raw")

axes[0, 1].scatter(temperature, sales_z, marker="x", color="C1")
axes[0, 1].set_xlabel("temperature")
axes[0, 1].set_ylabel("sales (z-score)")
axes[0, 1].set_title("z-score")

axes[1, 0].scatter(temperature[valid], log_sales[valid], marker="x", color="C2")
axes[1, 0].set_xlabel("temperature")
axes[1, 0].set_ylabel("log(sales)")
axes[1, 0].set_title("log")

axes[1, 1].scatter(temperature[valid], log_sales_z[valid], marker="x", color="C3")
axes[1, 1].set_xlabel("temperature")
axes[1, 1].set_ylabel("log(sales), z-score")
axes[1, 1].set_title("log + z-score: still the same curve")

for ax in axes.flat:
    ax.grid(True)

fig.suptitle("sales vs. temperature after each normalization", fontsize=14)
plt.tight_layout()
plt.show()

None of these transforms fix the underlying problem: `temperature` vs. each of the four normalized targets above still traces out the exact same non-linear (softplus-shaped) curve, only rescaled: normalizing $y$, however we choose to do it, cannot turn a non-linear conditional mean into a linear one. We still need a model whose mean is genuinely non-linear in `temperature`, i.e. one built from a link function, linear only in its parameters, not in the raw input.

We can model this kind of target with a generalized linear model with a link function that turns negative values into positive values. We could fit a plain linear model and then saturate negative outputs to $0$, but that would misspecify the noise: the mean here is generated as a smooth, strictly positive softplus function of temperature, plus homoscedastic Gaussian noise, so squared loss is exactly the right loss for this noise model: the only thing a plain linear model would get wrong is that, extrapolated to cold enough temperatures, it predicts a negative *mean*, which makes no sense for sales.

We will fit this same `sales`/`temperature` dataset with a linear model under ReLU, Softplus and Exponential link.


### Normalization strategy.

We normalize $\xvec$ (`temperature`) with a $z$-score, purely for numerical conditioning: with the exponential link in particular, an unnormalized input (up to $30$ here) can make $\exp(\xvect\wvec)$ overflow for an otherwise unremarkable random initialization of $\wvec$.

We do not $z$-score the target $\tvec$ (`sales`). A $z$-score recenters the target to have mean exactly $0$, so roughly half its values become negative. But ReLU, softplus and the exponential link only ever output values in $[0,\infty)$ or $(0,\infty)$: for every point whose $z$-scored target is negative, the model can at best saturate near its floor ($0$), never actually reach it, and no adjustment of $\wvec$ fixes that: it is a structural mismatch, not something more training fixes. Working around it would mean composing an extra affine rescaling on top of $\Phi$ (e.g. $a\,\Phi(\xvect\wvec)+c$) just to shift the output range back to where the target lives, at which point $\Phi$ is no longer doing the one job we wanted from it (keeping the *mean* positive), and we are back to needing extra free parameters outside the link.

Instead we min-max scale $\tvec$ to $[0,1]$: $\tvec_{\text{minmax}} = \dfrac{\tvec-\min(\tvec)}{\max(\tvec)-\min(\tvec)}$. This is still an affine (linear) transform of the data: subtracting a constant and scaling by another, so the additive Gaussian noise is still Gaussian afterwards (an affine map of a Gaussian is Gaussian). The difference with the $z$-score is *which* constant we subtract: the $z$-score subtracts the mean (forcing the transformed mean to $0$), while min-max subtracts the minimum (so the transformed mean lands somewhere inside $(0,1)$, never negative). Same kind of transform, different anchor point, and that's exactly what makes the min-max version compatible with a positive-mean link.

Here is a comparison of both normalization strategies:

In [ ]:
print(f"naive linear model at the coldest point would predict: {w_true*temperature.min()+b_true:.2f} (negative sales!)")
print(f"fraction of noisy observations that dip below 0: {np.mean(sales < 0):.0%}")

temperature_z = (temperature - temperature.mean()) / temperature.std()
sales_minmax = (sales - sales.min()) / (sales.max() - sales.min())

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

axes[0].scatter(temperature, sales, marker="x")
axes[0].set_xlabel("temperature")
axes[0].set_ylabel("sales")
axes[0].set_title("raw")

axes[1].scatter(temperature_z, sales_z, marker="x", color="C1")
axes[1].set_xlabel("temperature (z-score)")
axes[1].set_ylabel("sales (z-score)")
axes[1].set_title("both X and Y z-scored: Y goes negative")

axes[2].scatter(temperature_z, sales_minmax, marker="x", color="C2")
axes[2].set_xlabel("temperature (z-score)")
axes[2].set_ylabel("sales (min-max)")
axes[2].set_title("our strategy: X z-scored, Y min-max'd")

for ax in axes:
    ax.grid(True)

fig.suptitle("Data to be modeled: sales vs. temperature", fontsize=14)
plt.tight_layout()
plt.show()

### Function composition and Jacobian


Note that within a function composition, we just need to consider the elementwise mappings performed by the activation functions, with their corresponding gradients.

Following the same recipe as with the [SSE and absolute loss](1_Regression_Shallow_theory.ipynb), we express the squared-loss fit of $y=\Phi(\xvect\wvec)$ as a composition of vector functions, for $N$ training points: $\wvec\in\mathbb{R}^{D+1}$, $\Xmat\in\mathbb{R}^{N\times(D+1)}$, and $\tvec,\onevec\in\mathbb{R}^N$. The only new stage with respect to the plain SSE composition is the element-wise link $\Phi$ applied to the linear predictor, where $\zvec\in\mathbb{R}^N$ stacks, one per training point, the scalar linear predictor $x$ from the "Link functions" section above:

$$
\begin{align*}
\zvec &= \Xmat\wvec && \mathbb{R}^{D+1} \to \mathbb{R}^N\\
\yvec &= \Phi(\zvec) && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise}\\
\lvec &= (\tvec-\yvec)^2 && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise}\\
L &= \onevect\lvec && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

where Jacobians from each of the elements in the composition are given by:

$$
\begin{split}
J_\lvec &= \onevect \in \mathbb{R}^{1\times N}\\
J_\yvec &= -2\diag(\tvec-\yvec) \in \mathbb{R}^{N \times N}\\
J_{\zvec} &= \diag\pare{\Phi'(\zvec)} \in \mathbb{R}^{N \times N}\\
J_\wvec &= \Xmat \in \mathbb{R}^{N \times (D+1)}
\end{split}
$$

so the Jacobian of $L$ wrt $\wvec$ is:

$$
J_\wvec = -2\onevect\diag(\tvec-\yvec)\diag\pare{\Phi'(\zvec)}\Xmat
$$

and the gradient, its transpose:

$$
\nabla_\wvec = J_\wvec^T = -2\Xmatt\diag(\tvec-\yvec)\diag\pare{\Phi'(\zvec)}\onevec
$$

with $z^{(n)}={\xvect}^{(n)}\wvec$ and $y^{(n)}=\Phi(z^{(n)})$. Everything else is identical to the plain SSE case; the only change introduced by the link is the extra factor $\Phi'(z^{(n)})$ weighting each point's residual, exactly as the chain rule predicts.

### Specializing to ReLU, softplus and exponential

Since the only piece that changes across the three link functions is $\Phi'(\zvec)$, we just plug in the element-wise derivative of each (as usual, we set the ReLU's subgradient to $0$ at exactly $z=0$, same convention used for the absolute loss):

$$
\begin{array}{lll}
\text{ReLU:} & \Phi(z)=\max(0,z) & \Phi'(z) = \mathbb{1}[z>0]\\[4pt]
\text{Softplus:} & \Phi(z)=\log(1+e^{z}) & \Phi'(z) = \sigma(z) = \dfrac{1}{1+e^{-z}}\\[4pt]
\text{Exponential:} & \Phi(z)=e^{z} & \Phi'(z) = e^{z} = \Phi(z)
\end{array}
$$

Substituting into $\nabla_\wvec = -2\Xmatt\diag(\tvec-\yvec)\diag\pare{\Phi'(\zvec)}\onevec$:

$$
\begin{split}
\nabla_\wvec^{\text{ReLU}} &= -2\Xmatt\diag(\tvec-\yvec)\diag\pare{\mathbb{1}[\zvec>0]}\onevec\\
\nabla_\wvec^{\text{softplus}} &= -2\Xmatt\diag(\tvec-\yvec)\diag\pare{\sigma(\zvec)}\onevec\\
\nabla_\wvec^{\exp} &= -2\Xmatt\diag(\tvec-\yvec)\diag(\yvec)\onevec
\end{split}
$$

The ReLU gradient makes the earlier "dead unit" warning completely explicit: $\diag(\mathbb{1}[\zvec>0])$ zeroes out exactly the rows of $\tvec-\yvec$ belonging to points with $z^{(n)}\le 0$ (predicted exactly at the floor), so they contribute nothing to the gradient no matter how wrong their prediction is, that is the flat region of the loss surface, and it is why the fit can get stuck once enough points saturate there. Softplus never has this problem, since $\sigma(\zvec)>0$ everywhere. The exponential case is the tidiest of all: because $\Phi'=\Phi$, $\yvec$ itself doubles as the diagonal weighting matrix.

In [ ]:
## ===================================== ##
## ==== Gradient Descent, ReLU link ==== ##
## ===================================== ##
x_data_gd = temperature_z.reshape(-1, 1)
t_data_gd = sales_minmax.reshape(-1, 1)

np.random.seed(0)
w_relu, b_relu = create_computation_graph_linear(1, 1)

lr = 0.001
epochs = 5000
loss_history = []

for e in range(epochs):

    ## forward pass
    y_pred = computation_graph_relu(x_data_gd, w_relu, b_relu)
    loss_history.append(np.sum(squared_loss_function(t_data_gd, y_pred)))

    ## backward pass and update
    grad_w, grad_b = grad_squared_loss_wrt_relu_model(x_data_gd, t_data_gd, w_relu, b_relu)
    w_relu = w_relu - lr * grad_w
    b_relu = b_relu - lr * grad_b

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(loss_history)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Squared loss (sum)")
axes[0].set_title("Convergence: ReLU link")
axes[0].grid(True)

x_line = np.linspace(temperature_z.min(), temperature_z.max(), 200).reshape(-1, 1)
y_line = computation_graph_relu(x_line, w_relu, b_relu)

axes[1].scatter(temperature_z, sales_minmax, marker="x", label="data")
axes[1].plot(x_line, y_line, color="C1", linewidth=3, label=f"y = ReLU({w_relu.item():.3f}x + {b_relu.item():.3f})")
axes[1].set_xlabel("temperature (z-score)")
axes[1].set_ylabel("sales (min-max)")
axes[1].set_title("Fitted model: ReLU link")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

We observe ReLU, with this normalization, is not a good option since it saturates negative entries to zero. A min-max scaling on the X axis alone does not actually fix this: with $b$ left free, the fitted bias still ends up slightly negative to match the curve's true kink location, so we get essentially the same amount of saturation as with the z-score. What does fix it is constraining $\wvec,b\ge0$, which guarantees $z=\xvect\wvec\ge0$ for every point once $\xvec\ge0$ (as min-max scaling ensures): either by projecting $\wvec,b$ back onto $[0,\infty)$ after every gradient step, or more elegantly by passing them through a positive link function themselves. This would require one more step in the chain rule. **Exercise**

In [ ]:
## ========================================= ##
## ==== Gradient Descent, Softplus link ==== ##
## ========================================= ##
np.random.seed(0)
w_softplus, b_softplus = create_computation_graph_linear(1, 1)

lr = 0.01
epochs = 5000
loss_history = []

for e in range(epochs):

    ## forward pass
    y_pred = computation_graph_softplus(x_data_gd, w_softplus, b_softplus)
    loss_history.append(np.sum(squared_loss_function(t_data_gd, y_pred)))

    ## backward pass and update
    grad_w, grad_b = grad_squared_loss_wrt_softplus_model(x_data_gd, t_data_gd, w_softplus, b_softplus)
    w_softplus = w_softplus - lr * grad_w
    b_softplus = b_softplus - lr * grad_b

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(loss_history)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Squared loss (sum)")
axes[0].set_title("Convergence: Softplus link")
axes[0].grid(True)

x_line = np.linspace(temperature_z.min(), temperature_z.max(), 200).reshape(-1, 1)
y_line = computation_graph_softplus(x_line, w_softplus, b_softplus)

axes[1].scatter(temperature_z, sales_minmax, marker="x", label="data")
axes[1].plot(x_line, y_line, color="C1", linewidth=3, label=f"y = softplus({w_softplus.item():.3f}x + {b_softplus.item():.3f})")
axes[1].set_xlabel("temperature (z-score)")
axes[1].set_ylabel("sales (min-max)")
axes[1].set_title("Fitted model: Softplus link")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
## ============================================ ##
## ==== Gradient Descent, Exponential link ==== ##
## ============================================ ##
# even on the normalized scale, exp(w*x+b) grows fast enough that the default
# init (std=1) can overflow on the first step. A small init keeps z near 0.
np.random.seed(0)
w_exp, b_exp = create_computation_graph_linear(1, 1, mean=0, std=0.1)

lr = 0.001
epochs = 5000
loss_history = []

for e in range(epochs):

    ## forward pass
    y_pred = computation_graph_exponential(x_data_gd, w_exp, b_exp)
    loss_history.append(np.sum(squared_loss_function(t_data_gd, y_pred)))

    ## backward pass and update
    grad_w, grad_b = grad_squared_loss_wrt_exponential_model(x_data_gd, t_data_gd, w_exp, b_exp)
    w_exp = w_exp - lr * grad_w
    b_exp = b_exp - lr * grad_b

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(loss_history)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Squared loss (sum)")
axes[0].set_title("Convergence: Exponential link")
axes[0].grid(True)

x_line = np.linspace(temperature_z.min(), temperature_z.max(), 200).reshape(-1, 1)
y_line = computation_graph_exponential(x_line, w_exp, b_exp)

axes[1].scatter(temperature_z, sales_minmax, marker="x", label="data")
axes[1].plot(x_line, y_line, color="C1", linewidth=3, label=f"y = exp({w_exp.item():.3f}x + {b_exp.item():.3f})")
axes[1].set_xlabel("temperature (z-score)")
axes[1].set_ylabel("sales (min-max)")
axes[1].set_title("Fitted model: Exponential link")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Example: heteroscedastic regression with positive mean

Same generative story as before (temperature $\to$ softplus mean), but now the noise scale itself grows with the mean instead of staying constant; sales fluctuate more on busy days than on quiet ones.

In [ ]:
rng = np.random.default_rng(7)
n = 300
temperature_hetero = rng.uniform(-20, 30, size=n)

w_true, b_true = 0.15, 0.5
mean_sales_hetero = computation_graph_softplus(temperature_hetero.reshape(-1, 1), np.array([[w_true]]), np.array([[b_true]])).ravel()  # softplus mean, same as before

sigma0, sigma1 = 0.1, 0.15
sigma_sales = sigma0 + sigma1 * mean_sales_hetero  # heteroscedastic: noise scale grows with the mean

sales_hetero = mean_sales_hetero + rng.normal(0, 1, size=n) * sigma_sales

print(f"sigma ranges from {sigma_sales.min():.2f} to {sigma_sales.max():.2f} across the data")
print(f"fraction of noisy observations that dip below 0: {np.mean(sales_hetero < 0):.0%}")

# same normalization strategy as before: X z-scored, Y min-max'd
temperature_hetero_z = (temperature_hetero - temperature_hetero.mean()) / temperature_hetero.std()
sales_hetero_minmax = (sales_hetero - sales_hetero.min()) / (sales_hetero.max() - sales_hetero.min())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(temperature_hetero, sales_hetero, marker="x")
axes[0].set_xlabel("temperature")
axes[0].set_ylabel("sales")
axes[0].set_title("raw")

axes[1].scatter(temperature_hetero_z, sales_hetero_minmax, marker="x", color="C2")
axes[1].set_xlabel("temperature (z-score)")
axes[1].set_ylabel("sales (min-max)")
axes[1].set_title("normalized: X z-scored, Y min-max'd")

for ax in axes:
    ax.grid(True)

fig.suptitle("Heteroscedastic data to be modeled: sales vs. temperature", fontsize=14)
plt.tight_layout()
plt.show()

We will model this Gaussianly, but now letting the noise scale depend on the input, $\sigma^2(\xvec^{(n)})$, instead of a single constant $\sigma^2$ shared by every point:

$$
\begin{split}
p(t^n\mid {\xvect}^{(n)}\wvec,\sigma^2(\xvec^{(n)})) = \frac{1}{\sqrt{2\pi\sigma^2(\xvec^{(n)})}}\exp\pare{-\frac{(t^n-\mu(\xvec^{n}))^2}{2\sigma^2(\xvec^{(n)})}}
\end{split}
$$

with:

$$
\begin{split}
\mu(\xvec^{n}) = \text{softplus}\pare{{\xvect}^{(n)}\wvec}\\
\sigma^2(\xvec^{(n)}) = e^{{\xvect}^{(n)}\wvecsigma}
\end{split}
$$


so the log-likelihood, and thus the cost function, is:

$$
\begin{split}
\sum^N_{n=1}\log p(t^n\mid {\xvect}^{(n)}\wvec,\sigma^2(\xvec^{(n)})) = -\frac{N}{2}\log(2\pi) - \frac{1}{2}\sum^N_{n=1}\log\sigma^2(\xvec^{(n)}) - \frac{1}{2}\sum^N_{n=1}\frac{(t^n-\mu(\xvec^{n}))^2}{\sigma^2(\xvec^{(n)})}
\end{split}
$$

### Optimization algorithm

The coordinate descent algorithm (exact GLS step for $\wvec$, gradient step for $\wvec_\sigma$) derived in [3_Probabilistic_perspective.ipynb](3_Probabilistic_perspective.ipynb) does not apply here: that exact step for $\wvec$ assumes a plain linear mean, $\yvec=\Xmat\wvec$, but here the mean goes through the softplus link. So we fit both $\wvec$ and $\wvec_\sigma$ by plain gradient descent.

In [ ]:
x_data = temperature_hetero_z.reshape(-1, 1)
t_data = sales_hetero_minmax.reshape(-1, 1)

np.random.seed(0)
w, b = create_computation_graph_linear(1, 1)
w_sigma, b_sigma = create_computation_graph_linear(1, 1)

lr = 1e-4
epochs = 5000
loss_history = []

for e in range(epochs):

    ## forward pass
    y_pred, v_pred = computation_graph_heteroscedastic_gaussian_softplus(x_data, w, b, w_sigma, b_sigma)
    loss_history.append(np.sum(heteroscedastic_loss_function(t_data, y_pred, v_pred)))

    ## backward pass and update
    grad_w, grad_b, grad_w_sigma, grad_b_sigma = grad_heteroscedastic_loss_wrt_softplus_model(
        x_data, t_data, w, b, w_sigma, b_sigma)
    w = w - lr * grad_w
    b = b - lr * grad_b
    w_sigma = w_sigma - lr * grad_w_sigma
    b_sigma = b_sigma - lr * grad_b_sigma

x_line = np.linspace(x_data.min(), x_data.max(), 200).reshape(-1, 1)
y_line, v_line = computation_graph_heteroscedastic_gaussian_softplus(x_line, w, b, w_sigma, b_sigma)
sigma_line = np.sqrt(v_line)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(loss_history)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (heteroscedastic)")
axes[0].set_title("Convergence: gradient descent")
axes[0].grid(True)

axes[1].scatter(x_data, t_data, marker="x", label="data")
axes[1].plot(x_line, y_line, color="red", label="fitted mean (softplus)")
axes[1].fill_between(x_line.ravel(), (y_line - sigma_line).ravel(), (y_line + sigma_line).ravel(),
                      color="red", alpha=0.2, label="±1 std (heteroscedastic)")
axes[1].set_xlabel("temperature (z-score)")
axes[1].set_ylabel("sales (min-max)")
axes[1].set_title("Fitted mean and heteroscedastic uncertainty")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## TODO

* Talk about canonical links.
* Explain sometimes link functions are used not because they are the canonical but for mathematical convenience, such as Gaussian Process classification.